# Gemma / Llama-family fallback run

Use a GPU runtime. This notebook fixes the tokenizer-portability issue in the original smoke test, authenticates Hugging Face for gated Gemma access, prints complete failures, and continues until one non-Qwen model passes.

For Gemma, first accept the model license at https://huggingface.co/google/gemma-2-2b and provide a Hugging Face read token when prompted.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!python -m pip install -q --upgrade 'transformers>=4.45' accelerate sentencepiece protobuf huggingface_hub


In [ ]:
# Gemma is gated. Enter a read token for an account that has accepted its license.
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
%cd /content
!rm -rf lengthgen
!git clone https://github.com/arkankau/lengthgen.git
%cd /content/lengthgen
!git pull


In [ ]:
import os, subprocess

OUTROOT = '/content/drive/MyDrive/lengthgen_realmodel_family'
os.makedirs(OUTROOT, exist_ok=True)

# Prefer Gemma; then try base and chat TinyLlama checkpoints.
CANDIDATES = [
    ('gemma2b', 'google/gemma-2-2b'),
    ('tinyllama_base', 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'),
    ('tinyllama_chat', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'),
]

def run_visible(cmd):
    print('\n' + '=' * 100, flush=True)
    print(cmd, flush=True)
    print('=' * 100, flush=True)
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, flush=True)
    return result.returncode

working = None
for short, model in CANDIDATES:
    outdir = f'{OUTROOT}/{short}_smoke'
    code = run_visible(
        f'python colab/real_model_probe.py --model {model} ' 
        f'--smoke --batch 1 --dtype auto --outdir {outdir}'
    )
    if code == 0:
        working = (short, model)
        print('PASSED:', working)
        break
    print('FAILED:', model, 'exit code', code)

assert working is not None, 'No candidate passed; the printed traceback identifies the next fix.'


In [ ]:
short, model = working
outdir = f'{OUTROOT}/{short}_h8'
code = run_visible(
    f'python colab/real_model_probe.py --model {model} ' 
    f'--lengths 5,10,20,40,80,160 --n 150 --heads 8 --batch 1 --dtype auto ' 
    f'--outdir {outdir}'
)
assert code == 0, f'Full run failed with exit code {code}; partial lengths remain saved in {outdir}'


In [ ]:
!find /content/drive/MyDrive/lengthgen_realmodel_family -name realmodel_results.json -print
